# Parameter Estimation with Maximum Likelihood

A compact, Python-first study of point estimation using Exercise 6 and the exponential rate as the central example.

## 1. Estimation in one page

An **estimator** $\widehat\theta=T(X_1,\ldots,X_n)$ is a random variable; its observed numerical value is an **estimate**. Important performance measures are

$$\operatorname{Bias}(\widehat\theta)=E(\widehat\theta)-\theta,$$
$$\operatorname{SE}(\widehat\theta)=\sqrt{\operatorname{Var}(\widehat\theta)},$$
$$\operatorname{MSE}(\widehat\theta)=\operatorname{Var}(\widehat\theta)+\operatorname{Bias}(\widehat\theta)^2.$$

A confidence interval is a procedure whose repeated-sampling coverage is the stated confidence level; it is not a probability statement about a fixed parameter after the data have been observed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(2026)

## 2. Exercise 6: MLE for the exponential rate

Let $X_1,\ldots,X_n$ be an independent random sample from an exponential distribution with density

$$f(x;\theta)=\theta e^{-\theta x},\qquad x>0,\quad \theta>0,$$

where $\theta$ is the unknown **rate**. Find its maximum likelihood estimator. NumPy and SciPy use the scale parameter $1/\theta$ when generating exponential observations.

### Likelihood, log-likelihood, and analytical MLE

For an observed sample $x_1,\ldots,x_n>0$, independence gives

$$L(\theta;\mathbf{x})=\prod_{i=1}^n\theta e^{-\theta x_i}=\theta^n\exp\left(-\theta\sum_{i=1}^n x_i\right).$$

The log-likelihood is

$$\ell(\theta)=n\log\theta-\theta\sum_{i=1}^n x_i.$$

Differentiating and setting the score equal to zero,

$$\ell'(\theta)=\frac{n}{\theta}-\sum_{i=1}^n x_i=0\quad\Longrightarrow\quad\widehat\theta_{MLE}=\frac{n}{\sum_{i=1}^n x_i}=\frac{1}{\bar x}.$$

Because $\ell''(\theta)=-n/\theta^2<0$ for every $\theta>0$, this is the unique maximum:

$$\boxed{\widehat\theta_{MLE}=1/\bar X}.$$

### Reproducible numerical example

In [ ]:
theta_true = 2.0
n = 25

x = rng.exponential(scale=1 / theta_true, size=n)
theta_hat = n / x.sum()

pd.Series({
    "sample_size": n,
    "sample_mean": x.mean(),
    "true_rate": theta_true,
    "MLE_rate": theta_hat,
}).round(4)

### Numerical verification with `scipy.optimize`

Minimizing the negative log-likelihood should reproduce the closed-form estimator.

In [ ]:
def negative_log_likelihood(theta, sample):
    if theta <= 0:
        return np.inf
    return -(sample.size * np.log(theta) - theta * sample.sum())

result = minimize_scalar(
    negative_log_likelihood,
    bounds=(1e-8, 20.0),
    args=(x,),
    method="bounded",
)

pd.Series({
    "closed_form_MLE": theta_hat,
    "numerical_optimum": result.x,
    "absolute_difference": abs(theta_hat - result.x),
    "optimizer_success": result.success,
})

## 3. Finite-sample properties

Let $S=\sum_iX_i$. Since $S\sim\operatorname{Gamma}(n,\text{rate}=\theta)$ and $\widehat\theta=n/S$, for $n>2$:

$$E(\widehat\theta)=\frac{n\theta}{n-1},\qquad \operatorname{Bias}(\widehat\theta)=\frac{\theta}{n-1},$$
$$\operatorname{Var}(\widehat\theta)=\frac{n^2\theta^2}{(n-1)^2(n-2)},$$
$$\operatorname{SE}(\widehat\theta)=\frac{n\theta}{(n-1)\sqrt{n-2}},$$
$$\operatorname{MSE}(\widehat\theta)=\frac{\theta^2(n+2)}{(n-1)(n-2)}.$$

Thus the rate MLE is upward-biased for finite $n$, but its bias and variability decrease with sample size. The estimator $(n-1)/S$ is unbiased for $n>1$.

## 4. Exact confidence interval

The pivot $2\theta S$ follows $\chi^2_{2n}$. Therefore, an exact $(1-\alpha)$ confidence interval for the exponential rate is

$$\left[\frac{\chi^2_{2n,\alpha/2}}{2S},\frac{\chi^2_{2n,1-\alpha/2}}{2S}\right].$$

In [ ]:
def exponential_rate_ci(sample, confidence=0.95):
    sample = np.asarray(sample, dtype=float)
    if sample.size == 0 or np.any(sample <= 0):
        raise ValueError("sample must contain positive observations")
    alpha = 1 - confidence
    df = 2 * sample.size
    total = sample.sum()
    return (
        stats.chi2.ppf(alpha / 2, df) / (2 * total),
        stats.chi2.ppf(1 - alpha / 2, df) / (2 * total),
    )

ci_low, ci_high = exponential_rate_ci(x)
pd.Series({"MLE": theta_hat, "95% CI lower": ci_low, "95% CI upper": ci_high})

## 5. Small simulation study

Repeated sampling shows how the estimator's empirical bias, variance, standard error, and MSE approach their theoretical values as $n$ grows.

In [ ]:
sim_rng = np.random.default_rng(2026)
sample_sizes = np.array([5, 20, 100, 500])
repetitions = 10_000
estimates = {}
rows = []

for n_sim in sample_sizes:
    samples = sim_rng.exponential(scale=1 / theta_true, size=(repetitions, n_sim))
    theta_hats = n_sim / samples.sum(axis=1)
    estimates[n_sim] = theta_hats
    exact_bias = theta_true / (n_sim - 1)
    exact_variance = n_sim**2 * theta_true**2 / ((n_sim - 1)**2 * (n_sim - 2))
    exact_mse = theta_true**2 * (n_sim + 2) / ((n_sim - 1) * (n_sim - 2))
    rows.append({
        "n": n_sim,
        "empirical_bias": theta_hats.mean() - theta_true,
        "exact_bias": exact_bias,
        "empirical_variance": theta_hats.var(ddof=1),
        "exact_variance": exact_variance,
        "empirical_SE": theta_hats.std(ddof=1),
        "exact_SE": np.sqrt(exact_variance),
        "empirical_MSE": np.mean((theta_hats - theta_true) ** 2),
        "exact_MSE": exact_mse,
    })

simulation_results = pd.DataFrame(rows).set_index("n")
simulation_results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for n_sim in sample_sizes[:3]:
    axes[0].hist(estimates[n_sim], bins=80, density=True, histtype="step", linewidth=1.7, label=f"n = {n_sim}")
axes[0].axvline(theta_true, color="black", linestyle="--", label="true rate")
axes[0].set_xlim(0, np.quantile(estimates[5], 0.995))
axes[0].set(xlabel=r"$\widehat{\theta}$", ylabel="Density", title="Sampling distributions")
axes[0].legend()

empirical_means = [estimates[n_sim].mean() for n_sim in sample_sizes]
exact_means = theta_true * sample_sizes / (sample_sizes - 1)
axes[1].plot(sample_sizes, empirical_means, "o-", label="simulation mean")
axes[1].plot(sample_sizes, exact_means, "s--", label=r"exact $E(\widehat\theta)$")
axes[1].axhline(theta_true, color="black", linestyle=":", label="true rate")
axes[1].set_xscale("log")
axes[1].set(xlabel="Sample size n (log scale)", ylabel=r"Mean of $\widehat{\theta}$", title="Bias decreases as n grows")
axes[1].legend()

fig.tight_layout()
plt.show()

## 6. Conclusion

The analytical and numerical optimizations agree. The MLE $1/\bar X$ is consistent but upward-biased in finite samples, and the simulation reproduces its exact bias, variance, standard error, and MSE. The chi-square pivot provides an exact confidence interval for the unknown rate.